# 02 — Diagram Detect: find tags on the P&ID

Companion to [Chapter 08](../08-diagram-annotation.md). Cell order: auth ->
list/retrieve -> submit job -> poll **safely, with a bound** -> inspect result ->
create edges.

**Known platform caveat, verified in this training project:** detect jobs can get
stuck at `Distributed` with no cancel API. Never poll in an unbounded loop. If a job
of yours gets stuck, stop -- do not resubmit.

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive
import time
from cognite.client.data_classes.data_modeling import (
    DirectRelationReference,
    EdgeApply,
    NodeId,
    NodeOrEdgeData,
    ViewId,
)

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-diagram-detect")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import Row

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)


file_xid = f"file_{YOURNAME}_TRN_PID_21_SEP"
v_asset  = ASSET

print("connected:", client.config.project, "| space:", space)

## Step 1 -- build the entity list to search for

`diagrams.detect` needs candidate entity names to look for on the page -- your
assets, by name and externalId.

In [ ]:
assets = client.data_modeling.instances.list(instance_type="node", sources=[v_asset], space=space, limit=-1)

entities = []
for a in assets:
    props = a.properties.get(v_asset, {})
    name = props.get("name") or a.external_id
    entities.append({"externalId": a.external_id, "space": space, "name": [name, a.external_id]})

print(f"{len(entities)} candidate entities, e.g.: {entities[:2]}")

## Step 2 -- submit the detect job

In [ ]:
job = client.diagrams.detect(
    entities=entities,
    search_field="name",
    file_instance_ids=[NodeId(space, file_xid)],
    partial_match=True,
    min_tokens=2,
)
print("job submitted:", job)

## Step 3 -- poll SAFELY: bounded attempts, never an unbounded loop

`[COMMON MISTAKE]` an uncapped `while True: sleep(...)` is exactly how a stuck
`Distributed` job hangs your kernel indefinitely. Cap the attempts.

In [ ]:
MAX_ATTEMPTS = 20
SLEEP_SECONDS = 10

status = job.update_status()  # refreshes .status; bare job.status stays stale
for attempt in range(MAX_ATTEMPTS):
    print(f"attempt {attempt}: status={status}")
    if status in ("Completed", "Failed", "TimedOut"):
        break
    time.sleep(SLEEP_SECONDS)
    status = job.update_status()
else:
    print(
        f"Still not done after {MAX_ATTEMPTS} attempts. STOP -- do not resubmit. "
        "This matches the known 'stuck at Distributed' platform caveat -- escalate, don't retry."
    )



## Step 4 -- inspect the result and build edges

Only proceed past this cell if the previous cell actually reached `Completed`.

In [ ]:
# Prefer get_result() — returns {"items": [file_block, ...]}
result = job.get_result() if hasattr(job, "get_result") else (job.result if hasattr(job, "result") else job)

# items[] is one block PER FILE; detections live in block["annotations"]
annotations = []
if isinstance(result, dict):
    for block in result.get("items") or []:
        annotations.extend(block.get("annotations") or [])

print(f"{len(annotations)} raw detect hits")
for ann in annotations[:5]:
    print(
        " ",
        ann.get("text"),
        "conf=",
        ann.get("confidence"),
        "ents=",
        [(e.get("externalId") if isinstance(e, dict) else e) for e in (ann.get("entities") or [])],
    )



In [ ]:
def _bbox(region):
    verts = region.get("vertices") or []
    xs = [float(v["x"]) for v in verts if isinstance(v, dict) and "x" in v]
    ys = [float(v["y"]) for v in verts if isinstance(v, dict) and "y" in v]
    if xs and ys:
        return min(xs), max(xs), min(ys), max(ys)
    return 0.0, 0.1, 0.0, 0.1

v_anno = ViewId("cdf_cdm", "CogniteDiagramAnnotation", "v1")
edges = []
tags_found = []

for i, ann in enumerate(annotations):
    region = ann.get("region") or {}
    page = int(region.get("page") or ann.get("page") or 1)
    text = ann.get("text") or ""
    confidence = float(ann.get("confidence") or 0.0)
    x_min, x_max, y_min, y_max = _bbox(region)

    seen = set()  # API can list the same entity twice
    for ent in ann.get("entities") or []:
        asset_xid = ent.get("externalId") if isinstance(ent, dict) else str(ent)
        if not asset_xid or asset_xid in seen:
            continue
        seen.add(asset_xid)
        edge_xid = f"anno_{file_xid}_{asset_xid}_{i}"
        props = {
            "name": text or asset_xid,
            "confidence": confidence,
            "status": "Suggested",
            "startNodePageNumber": page,
            "startNodeText": text or asset_xid,
            "startNodeXMin": x_min,
            "startNodeXMax": x_max,
            "startNodeYMin": y_min,
            "startNodeYMax": y_max,
        }
        edges.append(EdgeApply(
            space=space,
            external_id=edge_xid,
            # Edge TYPE — not the view name. View CogniteDiagramAnnotation is in sources=.
            type=DirectRelationReference("cdf_cdm", "diagrams.AssetLink"),
            start_node=DirectRelationReference(space, file_xid),
            end_node=DirectRelationReference(space, asset_xid),
            sources=[NodeOrEdgeData(source=v_anno, properties=props)],
        ))
        if asset_xid not in tags_found:
            tags_found.append(asset_xid)

if edges:
    client.data_modeling.instances.apply(edges=edges)
print(f"created {len(edges)} annotation edge(s); tags found: {sorted(tags_found)}")



## section 8.7 -- read the edges back, from both ends

The edges exist. Now prove they are readable from *both* directions -- that is the
payoff for the `diagramAnnotations` connection declared on `Asset` in Chapter 03.

In [ ]:
# 1. The raw edges. instance_type="edge" is NOT optional: the default is "node",
#    and asking for nodes raises a 400 that names neither the parameter nor the view:
#      "A property from a node or edge only container was referenced in a context
#       where it is not allowed."
#    It means: CogniteDiagramAnnotation's container is usedFor: edge.
ANNOTATION = ViewId("cdf_cdm", "CogniteDiagramAnnotation", "v1")

anno_edges = client.data_modeling.instances.list(
    instance_type="edge", sources=ANNOTATION, space=space, limit=-1)

for e in anno_edges:
    pr = e.properties[ANNOTATION]
    print(f"{e.start_node.external_id:<30} -> {e.end_node.external_id:<14} "
          f"{str(pr.get('startNodeText')):<14} conf={pr.get('confidence')}")
print(f"\n{len(anno_edges)} annotation edge(s)")
# start_node = the P&ID file, end_node = the asset.
# That is why Asset.diagramAnnotations uses direction: inwards.

In [ ]:
# 2. Traverse it: from the pump, back along the edges, to the diagrams.
#    Three result-set expressions to say one English sentence.
q = Query(
    with_={
        "pump": NodeResultSetExpression(
            filter=flt.Equals(["node", "externalId"], "21-PA-2001A"), limit=1),
        "links": EdgeResultSetExpression(
            from_="pump", direction="inwards", limit=100,
            filter=flt.Equals(["edge", "type"],
                              {"space": "cdf_cdm", "externalId": "diagrams.AssetLink"})),
        "diagrams": NodeResultSetExpression(from_="links", limit=100),
    },
    select={
        "links": Select([SourceSelector(ANNOTATION, ["startNodeText", "confidence"])]),
        "diagrams": Select([SourceSelector(FILE, ["name"])]),
    },
)
res = client.data_modeling.instances.query(q)
print("annotations:", len(res["links"]), "| diagrams:", len(res["diagrams"]))
for n in res["diagrams"]:
    print("  ", n.properties[FILE].get("name"))
# annotations > 0 but diagrams == 0 means the edge points the wrong way.

## Verify in Fusion

Open the P&ID file (`file_<YOURNAME>_TRN_PID_21_SEP`) in Fusion's file viewer --
you should see a clickable bounding box over each tag found above.

## Bridge to the Function

Package this exact flow into `DetectDiagramTags`: env vars instead of a notebook
variable, the same bounded-poll discipline (a Function that hangs is far worse than a
notebook that hangs), and a returned dict (`annotations_created`, `tags_found`,
`tags_missing`) instead of printed output. See
[Chapter 08, section 8.5](../08-diagram-annotation.md#85-write-the-function-detectdiagramtags).